# Inventory and Production Performance — Safer Buffers

This notebook evaluates the **production decision after inventory is included** and builds a safer buffer policy without changing notebook `04` or its saved model policy.

It answers six operational questions:

1. How much production would the FIFO policy change?
2. How much recorded or simulated loss would remain?
3. Would the recommendation cover all **observed sales**?
4. Which stores and products create the largest operational risk?
5. How much safety is gained when buffers are increased to a service-level floor?
6. For a completed forecast date, how did the base forecast, safer buffer, inventory, production, sales, and loss compare?

The safer policy never lowers notebook `04`'s cost-selected buffer. For every store/product pair, it chooses the **smallest tested buffer** whose out-of-fold FIFO simulation reaches the configured observed-sales service target.

> **Important:** observed sales do not include customers who left because an item was sold out. Therefore, `MinimumUncoveredObservedSales` is a conservative lower bound, not a complete stockout estimate.

Run notebooks `01_data_preparation.ipynb` and `04_demand_model_ensemble_fifo.ipynb` first.


## 1. Setup


In [ ]:
from pathlib import Path
import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 120)

CURRENT_FOLDER = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        folder
        for folder in [CURRENT_FOLDER, *CURRENT_FOLDER.parents]
        if (folder / "src").exists() and (folder / "notebooks").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter inside the pastry-sales-forecasting project folder.")

DATA_MODE = os.getenv("PASTRY_DATA_MODE", "synthetic").strip().lower()
if DATA_MODE not in {"synthetic", "private"}:
    raise ValueError("PASTRY_DATA_MODE must be 'synthetic' or 'private'.")

if DATA_MODE == "private":
    PREPARED_FILE = PROJECT_ROOT / "data" / "private" / "processed" / "production.pkl"
    EVALUATION_FILE = PROJECT_ROOT / "data" / "private" / "processed" / "production_evaluation.pkl"
    OUTPUTS_PATH = PROJECT_ROOT / "outputs" / "private"
else:
    PREPARED_FILE = PROJECT_ROOT / "data" / "processed" / "production.pkl"
    EVALUATION_FILE = PROJECT_ROOT / "data" / "processed" / "production_evaluation.pkl"
    OUTPUTS_PATH = PROJECT_ROOT / "outputs"

FIFO_BUFFER_METRICS_PATH = OUTPUTS_PATH / "ensemble_fifo_buffer_metrics.csv"
FIFO_DETAILS_PATH = OUTPUTS_PATH / "ensemble_fifo_simulation_details.csv"
FIFO_POLICY_PATH = OUTPUTS_PATH / "ensemble_fifo_selected_policy.csv"

required_files = [PREPARED_FILE, FIFO_DETAILS_PATH, FIFO_POLICY_PATH]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Run notebooks 01 and 04 first. Missing files:\n- "
        + "\n- ".join(missing_files)
    )

SAFER_SERVICE_TARGET = float(
    os.getenv("PASTRY_SAFER_SERVICE_TARGET", "0.98")
)
if not 0.90 <= SAFER_SERVICE_TARGET < 1.0:
    raise ValueError(
        "PASTRY_SAFER_SERVICE_TARGET must be between 0.90 and 1.00."
    )

OUTPUTS_PATH.mkdir(parents=True, exist_ok=True)

print("Data mode:", DATA_MODE)
print("Prepared data:", PREPARED_FILE)
print("FIFO buffer simulations:", FIFO_DETAILS_PATH)
print("Safer observed-sales service target:", f"{SAFER_SERVICE_TARGET:.1%}")
print("Outputs:", OUTPUTS_PATH)


## 2. Load the cost-selected FIFO policy and recorded operations

Notebook `04` remains unchanged. Its cost-selected policy is loaded as the baseline, then notebook `05` derives a separate safer policy for evaluation.


In [ ]:
def normalize_boolean(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    normalized = series.astype("string").str.strip().str.lower()
    return normalized.isin({"true", "1", "yes", "y"})


production = pd.read_pickle(PREPARED_FILE)
production["Date"] = pd.to_datetime(production["Date"]).dt.normalize()

if FIFO_BUFFER_METRICS_PATH.exists():
    buffer_metrics = pd.read_csv(FIFO_BUFFER_METRICS_PATH)
    for column in ["SafetyBuffer", "TotalBusinessCostYen", "AbsoluteBuffer"]:
        if column not in buffer_metrics.columns:
            buffer_metrics[column] = np.nan
        buffer_metrics[column] = pd.to_numeric(
            buffer_metrics[column], errors="coerce"
        )
    valid_buffer_metrics = buffer_metrics.dropna(
        subset=["SafetyBuffer", "TotalBusinessCostYen"]
    ).copy()
else:
    valid_buffer_metrics = pd.DataFrame()

fifo_details = pd.read_csv(FIFO_DETAILS_PATH)
fifo_details["Date"] = pd.to_datetime(fifo_details["Date"]).dt.normalize()
fifo_details["SafetyBuffer"] = pd.to_numeric(
    fifo_details["SafetyBuffer"], errors="coerce"
).astype("Int64")

selected_policy = pd.read_csv(FIFO_POLICY_PATH)
if selected_policy.empty:
    selected_policy = pd.DataFrame(columns=["Store", "Product", "SafetyBuffer"])
else:
    selected_policy["SafetyBuffer"] = pd.to_numeric(
        selected_policy["SafetyBuffer"], errors="coerce"
    ).astype("Int64")

if not valid_buffer_metrics.empty:
    GLOBAL_FIFO_BUFFER = int(
        valid_buffer_metrics.sort_values(
            ["TotalBusinessCostYen", "AbsoluteBuffer"],
            na_position="last",
        ).iloc[0]["SafetyBuffer"]
    )
elif selected_policy["SafetyBuffer"].notna().any():
    GLOBAL_FIFO_BUFFER = int(selected_policy["SafetyBuffer"].median())
else:
    GLOBAL_FIFO_BUFFER = 0

selected_details = fifo_details.merge(
    selected_policy[["Store", "Product", "SafetyBuffer"]].rename(
        columns={"SafetyBuffer": "SelectedBuffer"}
    ),
    on=["Store", "Product"],
    how="left",
    validate="many_to_one",
)
selected_details["SelectedBuffer"] = (
    selected_details["SelectedBuffer"]
    .fillna(GLOBAL_FIFO_BUFFER)
    .astype(int)
)
selected_details = selected_details.loc[
    selected_details["SafetyBuffer"].astype(int).eq(
        selected_details["SelectedBuffer"]
    )
].copy()

if selected_details.empty:
    raise ValueError("No rows matched the selected FIFO policy.")

actual_columns = [
    "Date",
    "Store",
    "Product",
    "Production",
    "CarryoverStock",
    "Loss",
    "ClosingStock",
    "UseForInventoryModel",
    "Diagnostics",
]
for column in actual_columns:
    if column not in production.columns:
        production[column] = np.nan

recorded_operations = (
    production[actual_columns]
    .sort_values(["Date", "Store", "Product"])
    .drop_duplicates(["Date", "Store", "Product"], keep="last")
    .rename(
        columns={
            "Production": "ActualProduction",
            "CarryoverStock": "ActualOpeningCarryover",
            "Loss": "ActualLoss",
            "ClosingStock": "ActualClosingStock",
        }
    )
)

comparison = selected_details.merge(
    recorded_operations,
    on=["Date", "Store", "Product"],
    how="left",
    validate="one_to_one",
)

numeric_columns = [
    "ActualProduction",
    "ActualOpeningCarryover",
    "ActualLoss",
    "ActualClosingStock",
    "ActualDemand",
    "OpeningCarryover",
    "RecommendedProduction",
    "FulfilledSales",
    "StockoutUnits",
    "SimulatedLoss",
    "ClosingCarryover",
    "UnitCost",
    "UnitMarginEstimate",
    "SalesMarginYen",
    "LossCostYen",
    "SalesMinusLossCostYen",
]
for column in numeric_columns:
    comparison[column] = pd.to_numeric(comparison[column], errors="coerce")

comparison["UseForInventoryModel"] = normalize_boolean(
    comparison["UseForInventoryModel"]
)

print("Global fallback buffer:", GLOBAL_FIFO_BUFFER)
print("Cost-selected policy rows:", len(selected_details))
print("Rows marked clean for inventory comparison:", int(comparison["UseForInventoryModel"].sum()))
print("Rows excluded from direct comparison:", int((~comparison["UseForInventoryModel"]).sum()))
print(
    "Cost-selected policy period:",
    comparison["Date"].min().date(),
    "to",
    comparison["Date"].max().date(),
)


## 2.1 Build the safer buffer policy

The cost-selected buffer is profit/loss oriented. The safer policy adds a hard service constraint:

- keep the cost-selected buffer as the minimum;
- evaluate every tested FIFO buffer for each store/product pair;
- choose the smallest buffer reaching `PASTRY_SAFER_SERVICE_TARGET` (default **98%** observed-sales coverage);
- if the target is unreachable, use the largest tested buffer and flag the row.

This is safer than adding the same number everywhere because each product/store has different demand error and FIFO behaviour. Set `PASTRY_SAFER_SERVICE_TARGET=0.99` before running the notebook for a stricter policy, with more production and likely more loss.


In [ ]:
current_selected_details = selected_details.copy()


def summarise_buffer_candidates(details):
    summary = (
        details.groupby(
            ["Store", "Product", "SafetyBuffer"],
            as_index=False,
        )
        .agg(
            CalibrationRows=("Date", "size"),
            ActualDemand=("ActualDemand", "sum"),
            FulfilledSales=("FulfilledSales", "sum"),
            StockoutUnits=("StockoutUnits", "sum"),
            StockoutRows=("StockoutUnits", lambda values: values.gt(0).sum()),
            SimulatedLoss=("SimulatedLoss", "sum"),
            RecommendedProduction=("RecommendedProduction", "sum"),
            SalesMarginYen=("SalesMarginYen", "sum"),
            LossCostYen=("LossCostYen", "sum"),
            NetMarginAfterLossYen=("SalesMinusLossCostYen", "sum"),
        )
    )
    summary["ObservedSalesServiceLevel"] = np.where(
        summary["ActualDemand"].gt(0),
        summary["FulfilledSales"] / summary["ActualDemand"],
        1.0,
    )
    summary["StockoutRowRate"] = np.where(
        summary["CalibrationRows"].gt(0),
        summary["StockoutRows"] / summary["CalibrationRows"],
        np.nan,
    )
    return summary


buffer_candidates = summarise_buffer_candidates(fifo_details)
minimum_tested_buffer = int(buffer_candidates["SafetyBuffer"].min())
maximum_tested_buffer = int(buffer_candidates["SafetyBuffer"].max())

policy_keys = fifo_details[["Store", "Product"]].drop_duplicates()
current_buffer_policy = policy_keys.merge(
    selected_policy[["Store", "Product", "SafetyBuffer"]].rename(
        columns={"SafetyBuffer": "CurrentSafetyBuffer"}
    ),
    on=["Store", "Product"],
    how="left",
    validate="one_to_one",
)
current_buffer_policy["CurrentSafetyBuffer"] = (
    current_buffer_policy["CurrentSafetyBuffer"]
    .fillna(GLOBAL_FIFO_BUFFER)
    .clip(minimum_tested_buffer, maximum_tested_buffer)
    .astype(int)
)

safer_policy_rows = []
for policy_row in current_buffer_policy.itertuples(index=False):
    pair_candidates = buffer_candidates.loc[
        buffer_candidates["Store"].eq(policy_row.Store)
        & buffer_candidates["Product"].eq(policy_row.Product)
        & buffer_candidates["SafetyBuffer"].ge(
            policy_row.CurrentSafetyBuffer
        )
    ].sort_values("SafetyBuffer")

    if pair_candidates.empty:
        raise ValueError(
            f"No FIFO buffer candidates for {policy_row.Store} / "
            f"{policy_row.Product}."
        )

    target_candidates = pair_candidates.loc[
        pair_candidates["ObservedSalesServiceLevel"].ge(
            SAFER_SERVICE_TARGET
        )
    ]
    chosen = (
        target_candidates.iloc[0]
        if not target_candidates.empty
        else pair_candidates.iloc[-1]
    )

    current_match = pair_candidates.loc[
        pair_candidates["SafetyBuffer"].eq(
            policy_row.CurrentSafetyBuffer
        )
    ]
    current_metrics = (
        current_match.iloc[0]
        if not current_match.empty
        else pair_candidates.iloc[0]
    )

    safer_policy_rows.append(
        {
            "Store": policy_row.Store,
            "Product": policy_row.Product,
            "CurrentSafetyBuffer": int(
                policy_row.CurrentSafetyBuffer
            ),
            "SaferSafetyBuffer": int(chosen["SafetyBuffer"]),
            "BufferIncrease": int(
                chosen["SafetyBuffer"]
                - policy_row.CurrentSafetyBuffer
            ),
            "CalibrationRows": int(chosen["CalibrationRows"]),
            "CurrentServiceLevel": current_metrics[
                "ObservedSalesServiceLevel"
            ],
            "SaferServiceLevel": chosen[
                "ObservedSalesServiceLevel"
            ],
            "CurrentStockoutUnits": current_metrics[
                "StockoutUnits"
            ],
            "SaferStockoutUnits": chosen["StockoutUnits"],
            "CurrentSimulatedLoss": current_metrics[
                "SimulatedLoss"
            ],
            "SaferSimulatedLoss": chosen["SimulatedLoss"],
            "CurrentRecommendedProduction": current_metrics[
                "RecommendedProduction"
            ],
            "SaferRecommendedProduction": chosen[
                "RecommendedProduction"
            ],
            "TargetReached": bool(
                chosen["ObservedSalesServiceLevel"]
                >= SAFER_SERVICE_TARGET
            ),
        }
    )

safer_policy = pd.DataFrame(safer_policy_rows).sort_values(
    ["Store", "Product"]
).reset_index(drop=True)

selected_details = fifo_details.merge(
    safer_policy[["Store", "Product", "SaferSafetyBuffer"]],
    on=["Store", "Product"],
    how="inner",
    validate="many_to_one",
)
selected_details = selected_details.loc[
    selected_details["SafetyBuffer"].astype(int).eq(
        selected_details["SaferSafetyBuffer"]
    )
].copy()

if selected_details.empty:
    raise ValueError("No rows matched the safer FIFO policy.")

# Rebuild the comparison with the safer policy as the active policy.
comparison = selected_details.merge(
    recorded_operations,
    on=["Date", "Store", "Product"],
    how="left",
    validate="one_to_one",
)

numeric_columns = [
    "ActualProduction",
    "ActualOpeningCarryover",
    "ActualLoss",
    "ActualClosingStock",
    "ActualDemand",
    "OpeningCarryover",
    "RecommendedProduction",
    "FulfilledSales",
    "StockoutUnits",
    "SimulatedLoss",
    "ClosingCarryover",
    "UnitCost",
    "UnitMarginEstimate",
    "SalesMarginYen",
    "LossCostYen",
    "SalesMinusLossCostYen",
]
for column in numeric_columns:
    comparison[column] = pd.to_numeric(
        comparison[column], errors="coerce"
    )

comparison["UseForInventoryModel"] = normalize_boolean(
    comparison["UseForInventoryModel"]
)


def summarise_policy_validation(details, policy_name):
    actual_demand = details["ActualDemand"].sum()
    return {
        "Policy": policy_name,
        "AverageBuffer": details["SafetyBuffer"].mean(),
        "ObservedSales": actual_demand,
        "FulfilledSales": details["FulfilledSales"].sum(),
        "ObservedSalesServiceLevel": (
            details["FulfilledSales"].sum() / actual_demand
            if actual_demand
            else np.nan
        ),
        "StockoutUnits": details["StockoutUnits"].sum(),
        "StockoutRows": details["StockoutUnits"].gt(0).sum(),
        "RecommendedProduction": details[
            "RecommendedProduction"
        ].sum(),
        "SimulatedLoss": details["SimulatedLoss"].sum(),
        "NetMarginAfterLossYen": details[
            "SalesMinusLossCostYen"
        ].sum(min_count=1),
    }


buffer_policy_validation = pd.DataFrame(
    [
        summarise_policy_validation(
            current_selected_details,
            "Cost-selected policy from notebook 04",
        ),
        summarise_policy_validation(
            selected_details,
            f"Safer policy ({SAFER_SERVICE_TARGET:.0%} target)",
        ),
    ]
)

display(Markdown("### Buffer changes by store and product"))
display(
    safer_policy.style.format(
        {
            "CurrentSafetyBuffer": "{:,.0f}",
            "SaferSafetyBuffer": "{:,.0f}",
            "BufferIncrease": "{:+,.0f}",
            "CalibrationRows": "{:,.0f}",
            "CurrentServiceLevel": "{:.1%}",
            "SaferServiceLevel": "{:.1%}",
            "CurrentStockoutUnits": "{:,.0f}",
            "SaferStockoutUnits": "{:,.0f}",
            "CurrentSimulatedLoss": "{:,.0f}",
            "SaferSimulatedLoss": "{:,.0f}",
            "CurrentRecommendedProduction": "{:,.0f}",
            "SaferRecommendedProduction": "{:,.0f}",
        }
    )
)

display(Markdown("### Historical out-of-fold FIFO validation"))
display(
    buffer_policy_validation.style.format(
        {
            "AverageBuffer": "{:.1f}",
            "ObservedSales": "{:,.0f}",
            "FulfilledSales": "{:,.0f}",
            "ObservedSalesServiceLevel": "{:.1%}",
            "StockoutUnits": "{:,.0f}",
            "StockoutRows": "{:,.0f}",
            "RecommendedProduction": "{:,.0f}",
            "SimulatedLoss": "{:,.0f}",
            "NetMarginAfterLossYen": "¥{:,.0f}",
        }
    )
)

if not safer_policy["TargetReached"].all():
    display(
        Markdown(
            "**Warning:** at least one store/product pair could not reach "
            f"the {SAFER_SERVICE_TARGET:.1%} target with the buffers tested "
            "by notebook 04."
        )
    )

SAFER_POLICY_PATH = OUTPUTS_PATH / "inventory_safer_buffer_policy.csv"
safer_policy.to_csv(SAFER_POLICY_PATH, index=False)
print("Active policy: safer FIFO buffers")
print("Safer-policy rows:", len(selected_details))
print("Saved:", SAFER_POLICY_PATH)


## 2.2 Data-quality coverage and excluded rows

Rows with inconsistent stock movement stay available for diagnosis, but they are not used to judge the production policy directly. The coverage table shows how much of the safer-policy period is included.


In [ ]:
required_actual_fields = [
    "ActualProduction",
    "ActualOpeningCarryover",
    "ActualLoss",
    "ActualDemand",
]
comparison["RequiredActualFieldsComplete"] = comparison[
    required_actual_fields
].notna().all(axis=1)

comparison["UseForDirectComparison"] = (
    comparison["UseForInventoryModel"]
    & comparison["RequiredActualFieldsComplete"]
)

coverage_summary = pd.DataFrame(
    [
        {
            "SelectedPolicyRows": len(comparison),
            "CleanComparisonRows": int(comparison["UseForDirectComparison"].sum()),
            "ExcludedRows": int((~comparison["UseForDirectComparison"]).sum()),
            "ComparisonCoverage": comparison["UseForDirectComparison"].mean(),
            "RowsWithCompleteCostInformation": int(
                (
                    comparison["UnitCost"].notna()
                    & comparison["UnitMarginEstimate"].notna()
                ).sum()
            ),
        }
    ]
)

display(
    coverage_summary.style.format(
        {
            "SelectedPolicyRows": "{:,.0f}",
            "CleanComparisonRows": "{:,.0f}",
            "ExcludedRows": "{:,.0f}",
            "ComparisonCoverage": "{:.1%}",
            "RowsWithCompleteCostInformation": "{:,.0f}",
        }
    )
)

excluded_inventory_rows = comparison.loc[
    ~comparison["UseForDirectComparison"],
    [
        "Date",
        "Store",
        "Product",
        "ActualProduction",
        "ActualOpeningCarryover",
        "ActualLoss",
        "ActualClosingStock",
        "UseForInventoryModel",
        "RequiredActualFieldsComplete",
        "Diagnostics",
    ],
].copy()

if excluded_inventory_rows.empty:
    print("No inventory rows were excluded.")
else:
    diagnostic_text = excluded_inventory_rows["Diagnostics"].fillna("")
    diagnostic_patterns = {
        "Inventory does not balance": r"Inventory does not balance",
        "Sales source mismatch": r"Excel sales|outside Excel choux range",
        "Recorded loss differs from FIFO": r"Loss .* differs from FIFO expected",
        "Closing stock differs from FIFO": r"Closing stock .* differs from FIFO expected",
        "Loss exceeds opening carryover": r"Loss .* exceeds carryover stock",
        "Missing required actual field": r"^$",
    }
    diagnostic_counts = pd.DataFrame(
        [
            {
                "DiagnosticCategory": category,
                "Rows": int(diagnostic_text.str.contains(pattern, regex=True).sum()),
            }
            for category, pattern in diagnostic_patterns.items()
        ]
    ).query("Rows > 0").sort_values("Rows", ascending=False)
    display(Markdown("### Most frequent exclusion reasons"))
    display(diagnostic_counts)
    display(Markdown("### Sample excluded rows"))
    display(excluded_inventory_rows.head(30))

COVERAGE_PATH = OUTPUTS_PATH / "inventory_performance_coverage.csv"
EXCLUSIONS_PATH = OUTPUTS_PATH / "inventory_performance_excluded_rows.csv"
coverage_summary.to_csv(COVERAGE_PATH, index=False)
excluded_inventory_rows.to_csv(EXCLUSIONS_PATH, index=False)
print("Saved:", COVERAGE_PATH)
print("Saved:", EXCLUSIONS_PATH)


## 3. Build comparable operational results

Two views are kept separate:

- **Recorded operations:** observed sales, actual production, recorded loss, and recorded closing stock.
- **Safer FIFO policy:** the rolling simulation produced by notebook `04`.

For recorded operations, `MinimumUncoveredObservedSales` mainly detects inconsistent stock records. It is **not** proof that the shop achieved a 100% customer service level. For the policy simulation, it is the minimum number of observed sales that the recommendation would fail to cover.


In [ ]:
comparison["ActualAvailableStock"] = (
    comparison["ActualOpeningCarryover"].fillna(0)
    + comparison["ActualProduction"].fillna(0)
)
comparison["ActualObservedSalesCovered"] = np.minimum(
    comparison["ActualDemand"].fillna(0),
    comparison["ActualAvailableStock"],
)
comparison["ActualMinimumUncoveredObservedSales"] = np.maximum(
    comparison["ActualDemand"].fillna(0)
    - comparison["ActualAvailableStock"],
    0,
)

# Under FIFO, opening stock is sold before new production. Any opening stock
# that remains after observed sales is expected to expire at the end of the day.
comparison["FIFOExpectedLossFromRecordedFlow"] = np.maximum(
    comparison["ActualOpeningCarryover"].fillna(0)
    - comparison["ActualDemand"].fillna(0),
    0,
)
comparison["CalculatedActualClosingCarryover"] = np.maximum(
    comparison["ActualProduction"].fillna(0)
    - np.maximum(
        comparison["ActualDemand"].fillna(0)
        - comparison["ActualOpeningCarryover"].fillna(0),
        0,
    ),
    0,
)

comparison["ActualCostInformationComplete"] = (
    comparison["UnitCost"].notna()
    & comparison["UnitMarginEstimate"].notna()
    & comparison["UnitCost"].ge(0)
    & comparison["UnitMarginEstimate"].ge(0)
)
comparison["ActualSalesMarginYen"] = np.where(
    comparison["ActualCostInformationComplete"],
    comparison["ActualObservedSalesCovered"]
    * comparison["UnitMarginEstimate"],
    np.nan,
)
comparison["ActualLossCostYen"] = np.where(
    comparison["ActualCostInformationComplete"],
    comparison["ActualLoss"] * comparison["UnitCost"],
    np.nan,
)
comparison["ActualNetMarginAfterLossYen"] = (
    comparison["ActualSalesMarginYen"]
    - comparison["ActualLossCostYen"]
)

comparison["ProductionDifference"] = (
    comparison["RecommendedProduction"]
    - comparison["ActualProduction"]
)
comparison["LossDifference"] = (
    comparison["SimulatedLoss"]
    - comparison["ActualLoss"]
)
comparison["NetMarginAfterLossDifferenceYen"] = (
    comparison["SalesMinusLossCostYen"]
    - comparison["ActualNetMarginAfterLossYen"]
)

clean_comparison = comparison.loc[
    comparison["UseForDirectComparison"]
].copy()

if clean_comparison.empty:
    raise ValueError("No clean recorded inventory rows are available for comparison.")

print("Direct-comparison rows:", len(clean_comparison))
print(
    "Comparison period:",
    clean_comparison["Date"].min().date(),
    "to",
    clean_comparison["Date"].max().date(),
)
print(
    "Stores / products:",
    clean_comparison["Store"].nunique(),
    "/",
    clean_comparison[["Store", "Product"]].drop_duplicates().shape[0],
)


## 4. Overall operational KPIs

`ObservedSalesCoverage` is coverage of recorded sales only. Because lost customer demand is unknown, a policy with uncovered observed sales must be reviewed before operational use.


In [ ]:
def safe_ratio(numerator, denominator):
    return numerator / denominator if denominator else np.nan


def first_opening_stock(data, column):
    first_rows = (
        data.sort_values(["Date", "Store", "Product"])
        .drop_duplicates(["Store", "Product"], keep="first")
    )
    return first_rows[column].fillna(0).sum()


def operational_summary(data):
    if data.empty:
        return pd.DataFrame()

    observed_sales = data["ActualDemand"].sum()

    actual_stock_entering = (
        first_opening_stock(data, "ActualOpeningCarryover")
        + data["ActualProduction"].sum()
    )
    policy_stock_entering = (
        first_opening_stock(data, "OpeningCarryover")
        + data["RecommendedProduction"].sum()
    )

    rows = [
        {
            "Scenario": "Recorded operations",
            "Rows": len(data),
            "ObservedSales": observed_sales,
            "Production": data["ActualProduction"].sum(),
            "ObservedSalesCovered": data["ActualObservedSalesCovered"].sum(),
            "MinimumUncoveredObservedSales": data[
                "ActualMinimumUncoveredObservedSales"
            ].sum(),
            "ObservedSalesCoverage": safe_ratio(
                data["ActualObservedSalesCovered"].sum(), observed_sales
            ),
            "LossUnits": data["ActualLoss"].sum(),
            "LossRate": safe_ratio(data["ActualLoss"].sum(), actual_stock_entering),
            "AverageClosingCarryover": data["ActualClosingStock"].mean(),
            "SalesMarginYen": data["ActualSalesMarginYen"].sum(min_count=1),
            "LossCostYen": data["ActualLossCostYen"].sum(min_count=1),
            "NetMarginAfterLossYen": data[
                "ActualNetMarginAfterLossYen"
            ].sum(min_count=1),
            "CostCoverage": data["ActualCostInformationComplete"].mean(),
        },
        {
            "Scenario": "Safer FIFO policy",
            "Rows": len(data),
            "ObservedSales": observed_sales,
            "Production": data["RecommendedProduction"].sum(),
            "ObservedSalesCovered": data["FulfilledSales"].sum(),
            "MinimumUncoveredObservedSales": data["StockoutUnits"].sum(),
            "ObservedSalesCoverage": safe_ratio(
                data["FulfilledSales"].sum(), observed_sales
            ),
            "LossUnits": data["SimulatedLoss"].sum(),
            "LossRate": safe_ratio(data["SimulatedLoss"].sum(), policy_stock_entering),
            "AverageClosingCarryover": data["ClosingCarryover"].mean(),
            "SalesMarginYen": data["SalesMarginYen"].sum(min_count=1),
            "LossCostYen": data["LossCostYen"].sum(min_count=1),
            "NetMarginAfterLossYen": data[
                "SalesMinusLossCostYen"
            ].sum(min_count=1),
            "CostCoverage": data["CostInformationComplete"].fillna(False).mean(),
        },
    ]
    return pd.DataFrame(rows)


def build_delta_summary(summary, group_columns=None):
    group_columns = group_columns or []
    if summary.empty:
        return pd.DataFrame()

    metric_columns = [
        column
        for column in summary.columns
        if column not in [*group_columns, "Scenario"]
    ]

    actual = summary.loc[
        summary["Scenario"].eq("Recorded operations"),
        [*group_columns, *metric_columns],
    ].copy()
    policy = summary.loc[
        summary["Scenario"].eq("Safer FIFO policy"),
        [*group_columns, *metric_columns],
    ].copy()

    actual = actual.rename(
        columns={column: f"Actual{column}" for column in metric_columns}
    )
    policy = policy.rename(
        columns={column: f"Policy{column}" for column in metric_columns}
    )

    if group_columns:
        delta = actual.merge(policy, on=group_columns, how="inner", validate="one_to_one")
    else:
        actual["_key"] = 1
        policy["_key"] = 1
        delta = actual.merge(policy, on="_key", validate="one_to_one").drop(columns="_key")

    delta["ProductionChange"] = delta["PolicyProduction"] - delta["ActualProduction"]
    delta["ProductionChangePct"] = np.where(
        delta["ActualProduction"].ne(0),
        delta["ProductionChange"] / delta["ActualProduction"],
        np.nan,
    )
    delta["LossChange"] = delta["PolicyLossUnits"] - delta["ActualLossUnits"]
    delta["LossChangePct"] = np.where(
        delta["ActualLossUnits"].ne(0),
        delta["LossChange"] / delta["ActualLossUnits"],
        np.nan,
    )
    delta["ObservedSalesCoverageChange"] = (
        delta["PolicyObservedSalesCoverage"]
        - delta["ActualObservedSalesCoverage"]
    )
    delta["AverageClosingCarryoverChange"] = (
        delta["PolicyAverageClosingCarryover"]
        - delta["ActualAverageClosingCarryover"]
    )
    delta["NetMarginAfterLossChangeYen"] = (
        delta["PolicyNetMarginAfterLossYen"]
        - delta["ActualNetMarginAfterLossYen"]
    )

    def decision_flag(row):
        flags = []
        if row["PolicyMinimumUncoveredObservedSales"] > 0:
            flags.append("Observed-sales gap")
        if row["LossChange"] > 0:
            flags.append("More loss")
        if pd.notna(row["NetMarginAfterLossChangeYen"]) and row["NetMarginAfterLossChangeYen"] < 0:
            flags.append("Lower estimated margin")
        return "; ".join(flags) if flags else "No checked deterioration"

    delta["DecisionFlag"] = delta.apply(decision_flag, axis=1)
    return delta


overall_kpis = operational_summary(clean_comparison)
overall_delta = build_delta_summary(overall_kpis)

kpi_format = {
    "Rows": "{:,.0f}",
    "ObservedSales": "{:,.0f}",
    "Production": "{:,.0f}",
    "ObservedSalesCovered": "{:,.0f}",
    "MinimumUncoveredObservedSales": "{:,.0f}",
    "ObservedSalesCoverage": "{:.1%}",
    "LossUnits": "{:,.0f}",
    "LossRate": "{:.1%}",
    "AverageClosingCarryover": "{:.1f}",
    "SalesMarginYen": "¥{:,.0f}",
    "LossCostYen": "¥{:,.0f}",
    "NetMarginAfterLossYen": "¥{:,.0f}",
    "CostCoverage": "{:.1%}",
}

display(overall_kpis.style.format(kpi_format))

delta_columns = [
    "ProductionChange",
    "ProductionChangePct",
    "LossChange",
    "LossChangePct",
    "PolicyMinimumUncoveredObservedSales",
    "PolicyObservedSalesCoverage",
    "AverageClosingCarryoverChange",
    "NetMarginAfterLossChangeYen",
    "DecisionFlag",
]
display(
    overall_delta[delta_columns].style.format(
        {
            "ProductionChange": "{:+,.0f}",
            "ProductionChangePct": "{:+.1%}",
            "LossChange": "{:+,.0f}",
            "LossChangePct": "{:+.1%}",
            "PolicyMinimumUncoveredObservedSales": "{:,.0f}",
            "PolicyObservedSalesCoverage": "{:.1%}",
            "AverageClosingCarryoverChange": "{:+.1f}",
            "NetMarginAfterLossChangeYen": "¥{:+,.0f}",
        }
    )
)

result = overall_delta.iloc[0]
if result["PolicyMinimumUncoveredObservedSales"] > 0:
    decision_message = (
        "**REVIEW REQUIRED:** the safer policy reduces production/loss, "
        f"but it would fail to cover at least **{result['PolicyMinimumUncoveredObservedSales']:,.0f}** "
        "units of already observed sales. Because hidden lost demand is unknown, "
        "this policy should not be treated as deployment-ready without a stricter service constraint."
    )
else:
    decision_message = (
        "**Observed-sales coverage check passed:** the safer policy covers all "
        "recorded sales in the clean comparison rows. Hidden lost demand is still unknown."
    )
display(Markdown(decision_message))

OVERALL_PATH = OUTPUTS_PATH / "inventory_production_overall_kpis.csv"
OVERALL_DELTA_PATH = OUTPUTS_PATH / "inventory_production_overall_delta.csv"
overall_kpis.to_csv(OVERALL_PATH, index=False)
overall_delta.to_csv(OVERALL_DELTA_PATH, index=False)
print("Saved:", OVERALL_PATH)
print("Saved:", OVERALL_DELTA_PATH)


### How to read the overall difference

- Negative `ProductionChange` means less production than recorded operations.
- Negative `LossChange` means less simulated FIFO expiry than recorded loss.
- Positive `PolicyMinimumUncoveredObservedSales` means the policy would not have covered all recorded sales.
- Positive `NetMarginAfterLossChangeYen` means a better estimated sales margin after subtracting loss cost.
- A positive margin change does **not** automatically justify a policy that misses observed sales.


## 5. Performance by store and product

The delta tables put recorded operations and the safer policy on one row. The store/product risk table is sorted first by uncovered observed sales, then by the estimated margin change.


In [ ]:
def grouped_operational_summary(data, group_columns):
    frames = []
    for keys, group in data.groupby(group_columns, dropna=False, sort=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        summary = operational_summary(group)
        for column, value in reversed(list(zip(group_columns, keys))):
            summary.insert(0, column, value)
        frames.append(summary)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


by_store = grouped_operational_summary(clean_comparison, ["Store"])
by_product = grouped_operational_summary(clean_comparison, ["Product"])
by_store_product = grouped_operational_summary(
    clean_comparison, ["Store", "Product"]
)

by_store_delta = build_delta_summary(by_store, ["Store"])
by_product_delta = build_delta_summary(by_product, ["Product"])
by_store_product_delta = build_delta_summary(
    by_store_product, ["Store", "Product"]
)

comparison_columns = [
    "ActualObservedSales",
    "ActualProduction",
    "PolicyProduction",
    "ProductionChange",
    "ActualLossUnits",
    "PolicyLossUnits",
    "LossChange",
    "PolicyMinimumUncoveredObservedSales",
    "PolicyObservedSalesCoverage",
    "ActualAverageClosingCarryover",
    "PolicyAverageClosingCarryover",
    "NetMarginAfterLossChangeYen",
    "DecisionFlag",
]

group_format = {
    "ActualObservedSales": "{:,.0f}",
    "ActualProduction": "{:,.0f}",
    "PolicyProduction": "{:,.0f}",
    "ProductionChange": "{:+,.0f}",
    "ActualLossUnits": "{:,.0f}",
    "PolicyLossUnits": "{:,.0f}",
    "LossChange": "{:+,.0f}",
    "PolicyMinimumUncoveredObservedSales": "{:,.0f}",
    "PolicyObservedSalesCoverage": "{:.1%}",
    "ActualAverageClosingCarryover": "{:.1f}",
    "PolicyAverageClosingCarryover": "{:.1f}",
    "NetMarginAfterLossChangeYen": "¥{:+,.0f}",
}

if not by_store_delta.empty:
    display(Markdown("### By store"))
    display(
        by_store_delta[["Store", *comparison_columns]]
        .sort_values(
            ["PolicyMinimumUncoveredObservedSales", "NetMarginAfterLossChangeYen"],
            ascending=[False, True],
        )
        .style.format(group_format)
    )

if not by_product_delta.empty:
    display(Markdown("### By product"))
    display(
        by_product_delta[["Product", *comparison_columns]]
        .sort_values(
            ["PolicyMinimumUncoveredObservedSales", "NetMarginAfterLossChangeYen"],
            ascending=[False, True],
        )
        .style.format(group_format)
    )

if not by_store_product_delta.empty:
    display(Markdown("### Highest-risk store/product combinations"))
    risk_review = (
        by_store_product_delta[["Store", "Product", *comparison_columns]]
        .sort_values(
            ["PolicyMinimumUncoveredObservedSales", "NetMarginAfterLossChangeYen"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )
    display(risk_review.head(25).style.format(group_format))

by_store.to_csv(OUTPUTS_PATH / "inventory_production_by_store.csv", index=False)
by_product.to_csv(OUTPUTS_PATH / "inventory_production_by_product.csv", index=False)
by_store_product.to_csv(
    OUTPUTS_PATH / "inventory_production_by_store_product.csv", index=False
)
by_store_delta.to_csv(
    OUTPUTS_PATH / "inventory_production_by_store_delta.csv", index=False
)
by_product_delta.to_csv(
    OUTPUTS_PATH / "inventory_production_by_product_delta.csv", index=False
)
by_store_product_delta.to_csv(
    OUTPUTS_PATH / "inventory_production_by_store_product_delta.csv", index=False
)


## 6. Daily production, loss, and observed-sales coverage

Daily totals make it easier to find dates where production was reduced too aggressively or loss increased unexpectedly.


In [ ]:
daily_performance = (
    clean_comparison.groupby("Date", as_index=False)
    .agg(
        ObservedSales=("ActualDemand", "sum"),
        ActualProduction=("ActualProduction", "sum"),
        RecommendedProduction=("RecommendedProduction", "sum"),
        ActualLoss=("ActualLoss", "sum"),
        SimulatedLoss=("SimulatedLoss", "sum"),
        MinimumUncoveredObservedSales=("StockoutUnits", "sum"),
        ActualNetMarginAfterLossYen=("ActualNetMarginAfterLossYen", "sum"),
        PolicyNetMarginAfterLossYen=("SalesMinusLossCostYen", "sum"),
    )
)
daily_performance["ProductionChange"] = (
    daily_performance["RecommendedProduction"]
    - daily_performance["ActualProduction"]
)
daily_performance["LossChange"] = (
    daily_performance["SimulatedLoss"]
    - daily_performance["ActualLoss"]
)
daily_performance["ObservedSalesCoverage"] = np.where(
    daily_performance["ObservedSales"].ne(0),
    (
        daily_performance["ObservedSales"]
        - daily_performance["MinimumUncoveredObservedSales"]
    )
    / daily_performance["ObservedSales"],
    np.nan,
)
daily_performance["NetMarginAfterLossChangeYen"] = (
    daily_performance["PolicyNetMarginAfterLossYen"]
    - daily_performance["ActualNetMarginAfterLossYen"]
)

plt.figure(figsize=(12, 5))
plt.plot(
    daily_performance["Date"],
    daily_performance["ActualProduction"],
    marker="o",
    label="Actual production",
)
plt.plot(
    daily_performance["Date"],
    daily_performance["RecommendedProduction"],
    marker="o",
    label="Recommended production",
)
plt.title("Daily actual and recommended production")
plt.ylabel("Units")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(
    daily_performance["Date"],
    daily_performance["ActualLoss"],
    marker="o",
    label="Recorded loss",
)
plt.plot(
    daily_performance["Date"],
    daily_performance["SimulatedLoss"],
    marker="o",
    label="Safer-policy FIFO loss",
)
plt.title("Daily recorded loss and simulated FIFO loss")
plt.ylabel("Units")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

stockout_days = daily_performance.loc[
    daily_performance["MinimumUncoveredObservedSales"].gt(0)
].sort_values("MinimumUncoveredObservedSales", ascending=False)

if stockout_days.empty:
    display(Markdown("**No day in the clean period had an observed-sales coverage gap.**"))
else:
    display(Markdown("### Days requiring service-level review"))
    display(
        stockout_days.style.format(
            {
                "ObservedSales": "{:,.0f}",
                "ActualProduction": "{:,.0f}",
                "RecommendedProduction": "{:,.0f}",
                "ProductionChange": "{:+,.0f}",
                "ActualLoss": "{:,.0f}",
                "SimulatedLoss": "{:,.0f}",
                "LossChange": "{:+,.0f}",
                "MinimumUncoveredObservedSales": "{:,.0f}",
                "ObservedSalesCoverage": "{:.1%}",
                "ActualNetMarginAfterLossYen": "¥{:,.0f}",
                "PolicyNetMarginAfterLossYen": "¥{:,.0f}",
                "NetMarginAfterLossChangeYen": "¥{:+,.0f}",
            }
        )
    )

DAILY_PATH = OUTPUTS_PATH / "inventory_production_daily_performance.csv"
daily_performance.to_csv(DAILY_PATH, index=False)
print("Saved:", DAILY_PATH)


## 7. Review the latest completed forecast date

Only files whose names begin with a real ISO date are considered completed forecasts. This prevents `next_day_ensemble_fifo_forecast.csv` from being mistaken for a dated result.

Set the optional environment variable `PASTRY_REVIEW_DATE=YYYY-MM-DD` to review a specific completed date instead of the latest one.


In [ ]:
DATED_FORECAST_PATTERN = re.compile(
    r"^(?P<date>\d{4}-\d{2}-\d{2})_ensemble_fifo_forecast\.csv$"
)


def find_completed_forecasts(folder):
    completed = []
    for path in folder.glob("*_ensemble_fifo_forecast.csv"):
        match = DATED_FORECAST_PATTERN.match(path.name)
        if not match:
            continue
        try:
            forecast_date = pd.Timestamp(match.group("date")).normalize()
        except (TypeError, ValueError):
            continue
        completed.append((forecast_date, path))
    return sorted(completed, key=lambda item: item[0])


completed_forecasts = find_completed_forecasts(OUTPUTS_PATH)
requested_review_date = os.getenv("PASTRY_REVIEW_DATE", "").strip()

if not completed_forecasts:
    FORECAST_PATH = None
    forecast_date = None
    print("No completed dated forecast file was found. Run notebook 04 first.")
elif requested_review_date:
    requested_timestamp = pd.Timestamp(requested_review_date).normalize()
    matching = [
        (date, path)
        for date, path in completed_forecasts
        if date.eq(requested_timestamp)
    ]
    if not matching:
        available_dates = ", ".join(
            date.strftime("%Y-%m-%d") for date, _ in completed_forecasts
        )
        raise FileNotFoundError(
            f"No completed forecast exists for {requested_review_date}. "
            f"Available dates: {available_dates}"
        )
    forecast_date, FORECAST_PATH = matching[-1]
else:
    forecast_date, FORECAST_PATH = completed_forecasts[-1]

if FORECAST_PATH is not None:
    print("Completed forecast selected:", FORECAST_PATH)
    print("Forecast date:", forecast_date.date())


In [ ]:
if FORECAST_PATH is not None:
    forecast_review = pd.read_csv(FORECAST_PATH)

    numeric_review_columns = [
        "BaseForecastDemand",
        "SafetyBuffer",
        "ForecastDemand",
        "OpeningCarryover",
        "RecommendedProduction",
        "ActualProduction",
        "ActualSales",
        "ActualLoss",
        "ExpectedFIFOLoss",
    ]
    for column in numeric_review_columns:
        if column not in forecast_review.columns:
            forecast_review[column] = np.nan
        forecast_review[column] = pd.to_numeric(
            forecast_review[column], errors="coerce"
        )


    # Preserve notebook 04's dated recommendation, then apply the safer policy
    # only inside this review notebook.
    forecast_review["OriginalSafetyBuffer"] = forecast_review[
        "SafetyBuffer"
    ]
    forecast_review["OriginalForecastDemand"] = forecast_review[
        "ForecastDemand"
    ]
    forecast_review["OriginalRecommendedProduction"] = forecast_review[
        "RecommendedProduction"
    ]

    forecast_review = forecast_review.merge(
        safer_policy[["Store", "Product", "SaferSafetyBuffer"]],
        on=["Store", "Product"],
        how="left",
        validate="one_to_one",
    )
    forecast_review["SafetyBuffer"] = (
        forecast_review["SaferSafetyBuffer"]
        .combine_first(forecast_review["OriginalSafetyBuffer"])
        .fillna(GLOBAL_FIFO_BUFFER)
    )
    forecast_review["ForecastDemand"] = np.ceil(
        forecast_review["BaseForecastDemand"].fillna(0)
        + forecast_review["SafetyBuffer"].fillna(0)
    ).clip(lower=0)
    forecast_review["RecommendedProduction"] = np.maximum(
        forecast_review["ForecastDemand"]
        - forecast_review["OpeningCarryover"].fillna(0),
        0,
    )
    forecast_review["BufferIncrease"] = (
        forecast_review["SafetyBuffer"]
        - forecast_review["OriginalSafetyBuffer"]
    )
    forecast_review["ExtraProductionForSafety"] = (
        forecast_review["RecommendedProduction"]
        - forecast_review["OriginalRecommendedProduction"]
    )

    economics_columns = [
        "Date",
        "Store",
        "Product",
        "UnitCost",
        "UnitMarginEstimate",
        "ClosingStock",
    ]
    economics_rows = pd.DataFrame(
        columns=[
            "Store",
            "Product",
            "UnitCost",
            "UnitMarginEstimate",
            "ActualClosingStock",
        ]
    )

    if EVALUATION_FILE.exists():
        evaluation_data = pd.read_pickle(EVALUATION_FILE)
        evaluation_data["Date"] = pd.to_datetime(
            evaluation_data["Date"]
        ).dt.normalize()
        for column in economics_columns:
            if column not in evaluation_data.columns:
                evaluation_data[column] = np.nan
        economics_rows = (
            evaluation_data.loc[
                evaluation_data["Date"].eq(forecast_date),
                economics_columns,
            ]
            .drop_duplicates(["Store", "Product"], keep="last")
            .rename(columns={"ClosingStock": "ActualClosingStock"})
            .drop(columns="Date")
        )

    forecast_review = forecast_review.drop(
        columns=[
            column
            for column in ["UnitCost", "UnitMarginEstimate", "ActualClosingStock"]
            if column in forecast_review.columns
        ]
    ).merge(
        economics_rows,
        on=["Store", "Product"],
        how="left",
        validate="one_to_one",
    )

    for column in ["UnitCost", "UnitMarginEstimate", "ActualClosingStock"]:
        forecast_review[column] = pd.to_numeric(
            forecast_review[column], errors="coerce"
        )

    forecast_review["BaseForecastError"] = (
        forecast_review["BaseForecastDemand"]
        - forecast_review["ActualSales"]
    )
    forecast_review["BaseForecastAbsoluteError"] = forecast_review[
        "BaseForecastError"
    ].abs()
    forecast_review["BufferedTargetDifference"] = (
        forecast_review["ForecastDemand"]
        - forecast_review["ActualSales"]
    )

    forecast_review["ActualAvailableStock"] = (
        forecast_review["OpeningCarryover"].fillna(0)
        + forecast_review["ActualProduction"].fillna(0)
    )
    forecast_review["ActualMinimumUncoveredObservedSales"] = np.maximum(
        forecast_review["ActualSales"].fillna(0)
        - forecast_review["ActualAvailableStock"],
        0,
    )

    forecast_review["RecommendedAvailableStock"] = (
        forecast_review["OpeningCarryover"].fillna(0)
        + forecast_review["RecommendedProduction"].fillna(0)
    )

    forecast_review["OriginalRecommendedAvailableStock"] = (
        forecast_review["OpeningCarryover"].fillna(0)
        + forecast_review["OriginalRecommendedProduction"].fillna(0)
    )
    forecast_review["OriginalEstimatedFulfilledObservedSales"] = np.minimum(
        forecast_review["ActualSales"],
        forecast_review["OriginalRecommendedAvailableStock"],
    )
    forecast_review["OriginalMinimumUncoveredObservedSales"] = np.maximum(
        forecast_review["ActualSales"].fillna(0)
        - forecast_review["OriginalRecommendedAvailableStock"],
        0,
    )
    forecast_review["EstimatedFulfilledObservedSales"] = np.minimum(
        forecast_review["ActualSales"],
        forecast_review["RecommendedAvailableStock"],
    )
    forecast_review["MinimumUncoveredObservedSales"] = np.maximum(
        forecast_review["ActualSales"].fillna(0)
        - forecast_review["RecommendedAvailableStock"],
        0,
    )

    opening_already_covers_sales = (
        forecast_review["OpeningCarryover"].fillna(0)
        >= forecast_review["ActualSales"].fillna(0)
    )
    forecast_review["RequiredBufferForObservedSales"] = np.where(
        opening_already_covers_sales,
        0,
        np.maximum(
            forecast_review["ActualSales"].fillna(0)
            - forecast_review["BaseForecastDemand"].fillna(0),
            0,
        ),
    )
    forecast_review["AdditionalBufferNeededAfterSafer"] = np.maximum(
        forecast_review["RequiredBufferForObservedSales"]
        - forecast_review["SafetyBuffer"].fillna(0),
        0,
    )

    forecast_review["CalculatedActualClosingCarryover"] = np.maximum(
        forecast_review["ActualProduction"].fillna(0)
        - np.maximum(
            forecast_review["ActualSales"].fillna(0)
            - forecast_review["OpeningCarryover"].fillna(0),
            0,
        ),
        0,
    )
    forecast_review["ActualClosingCarryoverForReview"] = forecast_review[
        "ActualClosingStock"
    ].combine_first(forecast_review["CalculatedActualClosingCarryover"])

    forecast_review["RecommendedClosingCarryover"] = np.maximum(
        forecast_review["RecommendedProduction"].fillna(0)
        - np.maximum(
            forecast_review["ActualSales"].fillna(0)
            - forecast_review["OpeningCarryover"].fillna(0),
            0,
        ),
        0,
    )
    forecast_review["ProductionDifference"] = (
        forecast_review["RecommendedProduction"]
        - forecast_review["ActualProduction"]
    )
    forecast_review["LossDifferenceFromFIFOExpectation"] = (
        forecast_review["ActualLoss"]
        - forecast_review["ExpectedFIFOLoss"]
    )

    cost_complete = (
        forecast_review["UnitCost"].notna()
        & forecast_review["UnitMarginEstimate"].notna()
        & forecast_review["UnitCost"].ge(0)
        & forecast_review["UnitMarginEstimate"].ge(0)
    )
    forecast_review["ActualSalesMarginYen"] = np.where(
        cost_complete,
        forecast_review["ActualSales"]
        * forecast_review["UnitMarginEstimate"],
        np.nan,
    )
    forecast_review["ActualLossCostYen"] = np.where(
        cost_complete,
        forecast_review["ActualLoss"] * forecast_review["UnitCost"],
        np.nan,
    )
    forecast_review["ActualNetMarginAfterLossYen"] = (
        forecast_review["ActualSalesMarginYen"]
        - forecast_review["ActualLossCostYen"]
    )
    forecast_review["RecommendedSalesMarginYen"] = np.where(
        cost_complete,
        forecast_review["EstimatedFulfilledObservedSales"]
        * forecast_review["UnitMarginEstimate"],
        np.nan,
    )
    forecast_review["RecommendedLossCostYen"] = np.where(
        cost_complete,
        forecast_review["ExpectedFIFOLoss"]
        * forecast_review["UnitCost"],
        np.nan,
    )
    forecast_review["RecommendedNetMarginAfterLossYen"] = (
        forecast_review["RecommendedSalesMarginYen"]
        - forecast_review["RecommendedLossCostYen"]
    )
    forecast_review["NetMarginAfterLossDifferenceYen"] = (
        forecast_review["RecommendedNetMarginAfterLossYen"]
        - forecast_review["ActualNetMarginAfterLossYen"]
    )

    completed_rows = forecast_review.loc[
        forecast_review["ActualSales"].notna()
    ].copy()

    display(
        Markdown(
            f"### Forecast, inventory, and production review — "
            f"{forecast_date.strftime('%Y-%m-%d')}"
        )
    )

    if completed_rows.empty:
        print(
            "Actual sales are not available yet. The table contains the "
            "forecast and recommendation only."
        )
        display(
            forecast_review[
                [
                    "Store",
                    "Product",
                    "BaseForecastDemand",
                    "OriginalSafetyBuffer",
                    "SafetyBuffer",
                    "BufferIncrease",
                    "ForecastDemand",
                    "OpeningCarryover",
                    "OriginalRecommendedProduction",
                    "RecommendedProduction",
                    "ExtraProductionForSafety",
                ]
            ]
        )
    else:
        review_columns = [
            "Store",
            "Product",
            "BaseForecastDemand",
            "BaseForecastError",
            "OriginalSafetyBuffer",
            "SafetyBuffer",
            "BufferIncrease",
            "ForecastDemand",
            "OpeningCarryover",
            "ActualProduction",
            "OriginalRecommendedProduction",
            "RecommendedProduction",
            "ExtraProductionForSafety",
            "ProductionDifference",
            "ActualSales",
            "OriginalMinimumUncoveredObservedSales",
            "MinimumUncoveredObservedSales",
            "RequiredBufferForObservedSales",
            "AdditionalBufferNeededAfterSafer",
            "ActualLoss",
            "ExpectedFIFOLoss",
            "LossDifferenceFromFIFOExpectation",
            "ActualClosingCarryoverForReview",
            "RecommendedClosingCarryover",
            "NetMarginAfterLossDifferenceYen",
            "LossReason",
        ]
        review_columns = [
            column for column in review_columns if column in completed_rows.columns
        ]
        display(
            completed_rows
            .sort_values(
                ["MinimumUncoveredObservedSales", "BaseForecastAbsoluteError"],
                ascending=[False, False],
            )[review_columns]
            .style.format(
                {
                    "BaseForecastDemand": "{:,.0f}",
                    "BaseForecastError": "{:+,.0f}",
                    "OriginalSafetyBuffer": "{:+,.0f}",
                    "SafetyBuffer": "{:+,.0f}",
                    "BufferIncrease": "{:+,.0f}",
                    "ForecastDemand": "{:,.0f}",
                    "OpeningCarryover": "{:,.0f}",
                    "ActualProduction": "{:,.0f}",
                    "OriginalRecommendedProduction": "{:,.0f}",
                    "RecommendedProduction": "{:,.0f}",
                    "ExtraProductionForSafety": "{:+,.0f}",
                    "ProductionDifference": "{:+,.0f}",
                    "ActualSales": "{:,.0f}",
                    "OriginalMinimumUncoveredObservedSales": "{:,.0f}",
                    "MinimumUncoveredObservedSales": "{:,.0f}",
                    "RequiredBufferForObservedSales": "{:,.0f}",
                    "AdditionalBufferNeededAfterSafer": "{:,.0f}",
                    "ActualLoss": "{:,.0f}",
                    "ExpectedFIFOLoss": "{:,.0f}",
                    "LossDifferenceFromFIFOExpectation": "{:+,.0f}",
                    "ActualClosingCarryoverForReview": "{:,.0f}",
                    "RecommendedClosingCarryover": "{:,.0f}",
                    "NetMarginAfterLossDifferenceYen": "¥{:+,.0f}",
                }
            )
        )


In [ ]:
if FORECAST_PATH is not None and not completed_rows.empty:
    actual_sales_total = completed_rows["ActualSales"].sum()
    base_forecast_wape = safe_ratio(
        completed_rows["BaseForecastAbsoluteError"].sum(),
        actual_sales_total,
    )
    original_recommended_covered_sales = completed_rows[
        "OriginalEstimatedFulfilledObservedSales"
    ].sum()
    recommended_covered_sales = completed_rows[
        "EstimatedFulfilledObservedSales"
    ].sum()

    date_summary = pd.DataFrame(
        [
            {
                "CompletedRows": len(completed_rows),
                "ActualSales": actual_sales_total,
                "BaseForecastDemand": completed_rows[
                    "BaseForecastDemand"
                ].sum(),
                "BaseForecastWAPE": base_forecast_wape,
                "ForecastDemandIncludingBuffer": completed_rows[
                    "ForecastDemand"
                ].sum(),
                "ActualProduction": completed_rows[
                    "ActualProduction"
                ].sum(min_count=1),
                "OriginalRecommendedProduction": completed_rows[
                    "OriginalRecommendedProduction"
                ].sum(),
                "RecommendedProduction": completed_rows[
                    "RecommendedProduction"
                ].sum(),
                "ExtraProductionForSafety": completed_rows[
                    "ExtraProductionForSafety"
                ].sum(),
                "ProductionChange": (
                    completed_rows["RecommendedProduction"].sum()
                    - completed_rows["ActualProduction"].sum(min_count=1)
                ),
                "OriginalObservedSalesCoverage": safe_ratio(
                    original_recommended_covered_sales,
                    actual_sales_total,
                ),
                "RecommendedObservedSalesCoverage": safe_ratio(
                    recommended_covered_sales,
                    actual_sales_total,
                ),
                "OriginalMinimumUncoveredObservedSales": completed_rows[
                    "OriginalMinimumUncoveredObservedSales"
                ].sum(),
                "MinimumUncoveredObservedSales": completed_rows[
                    "MinimumUncoveredObservedSales"
                ].sum(),
                "ActualLoss": completed_rows["ActualLoss"].sum(min_count=1),
                "FIFOExpectedLoss": completed_rows[
                    "ExpectedFIFOLoss"
                ].sum(min_count=1),
                "ActualNetMarginAfterLossYen": completed_rows[
                    "ActualNetMarginAfterLossYen"
                ].sum(min_count=1),
                "RecommendedNetMarginAfterLossYen": completed_rows[
                    "RecommendedNetMarginAfterLossYen"
                ].sum(min_count=1),
            }
        ]
    )
    date_summary["NetMarginAfterLossDifferenceYen"] = (
        date_summary["RecommendedNetMarginAfterLossYen"]
        - date_summary["ActualNetMarginAfterLossYen"]
    )

    display(Markdown("### Completed-date summary"))
    display(
        date_summary.style.format(
            {
                "CompletedRows": "{:,.0f}",
                "ActualSales": "{:,.0f}",
                "BaseForecastDemand": "{:,.0f}",
                "BaseForecastWAPE": "{:.1%}",
                "ForecastDemandIncludingBuffer": "{:,.0f}",
                "ActualProduction": "{:,.0f}",
                "OriginalRecommendedProduction": "{:,.0f}",
                "RecommendedProduction": "{:,.0f}",
                "ExtraProductionForSafety": "{:+,.0f}",
                "ProductionChange": "{:+,.0f}",
                "OriginalObservedSalesCoverage": "{:.1%}",
                "RecommendedObservedSalesCoverage": "{:.1%}",
                "OriginalMinimumUncoveredObservedSales": "{:,.0f}",
                "OriginalMinimumUncoveredObservedSales": "{:,.0f}",
                "MinimumUncoveredObservedSales": "{:,.0f}",
                "ActualLoss": "{:,.0f}",
                "FIFOExpectedLoss": "{:,.0f}",
                "ActualNetMarginAfterLossYen": "¥{:,.0f}",
                "RecommendedNetMarginAfterLossYen": "¥{:,.0f}",
                "NetMarginAfterLossDifferenceYen": "¥{:+,.0f}",
            }
        )
    )

    date_by_store = (
        completed_rows.groupby("Store", as_index=False)
        .agg(
            ActualSales=("ActualSales", "sum"),
            BaseForecastDemand=("BaseForecastDemand", "sum"),
            BaseForecastAbsoluteError=("BaseForecastAbsoluteError", "sum"),
            ActualProduction=("ActualProduction", "sum"),
            OriginalRecommendedProduction=(
                "OriginalRecommendedProduction", "sum"
            ),
            RecommendedProduction=("RecommendedProduction", "sum"),
            ExtraProductionForSafety=("ExtraProductionForSafety", "sum"),
            OriginalMinimumUncoveredObservedSales=(
                "OriginalMinimumUncoveredObservedSales", "sum"
            ),
            MinimumUncoveredObservedSales=(
                "MinimumUncoveredObservedSales",
                "sum",
            ),
            ActualLoss=("ActualLoss", "sum"),
            FIFOExpectedLoss=("ExpectedFIFOLoss", "sum"),
            NetMarginAfterLossDifferenceYen=(
                "NetMarginAfterLossDifferenceYen",
                "sum",
            ),
        )
    )
    date_by_store["BaseForecastWAPE"] = np.where(
        date_by_store["ActualSales"].ne(0),
        date_by_store["BaseForecastAbsoluteError"]
        / date_by_store["ActualSales"],
        np.nan,
    )
    date_by_store["ProductionChange"] = (
        date_by_store["RecommendedProduction"]
        - date_by_store["ActualProduction"]
    )

    display(Markdown("### Completed date by store"))
    display(
        date_by_store[
            [
                "Store",
                "ActualSales",
                "BaseForecastDemand",
                "BaseForecastWAPE",
                "ActualProduction",
                "OriginalRecommendedProduction",
                "RecommendedProduction",
                "ExtraProductionForSafety",
                "ProductionChange",
                "OriginalMinimumUncoveredObservedSales",
                "MinimumUncoveredObservedSales",
                "ActualLoss",
                "FIFOExpectedLoss",
                "NetMarginAfterLossDifferenceYen",
            ]
        ]
        .sort_values("MinimumUncoveredObservedSales", ascending=False)
        .style.format(
            {
                "ActualSales": "{:,.0f}",
                "BaseForecastDemand": "{:,.0f}",
                "BaseForecastWAPE": "{:.1%}",
                "ActualProduction": "{:,.0f}",
                "OriginalRecommendedProduction": "{:,.0f}",
                "RecommendedProduction": "{:,.0f}",
                "ExtraProductionForSafety": "{:+,.0f}",
                "ProductionChange": "{:+,.0f}",
                "OriginalMinimumUncoveredObservedSales": "{:,.0f}",
                "MinimumUncoveredObservedSales": "{:,.0f}",
                "ActualLoss": "{:,.0f}",
                "FIFOExpectedLoss": "{:,.0f}",
                "NetMarginAfterLossDifferenceYen": "¥{:+,.0f}",
            }
        )
    )

    minimum_gap = date_summary.loc[0, "MinimumUncoveredObservedSales"]
    original_gap = date_summary.loc[
        0, "OriginalMinimumUncoveredObservedSales"
    ]
    if minimum_gap > 0:
        largest_extra_buffer = completed_rows[
            "AdditionalBufferNeededAfterSafer"
        ].max()
        display(
            Markdown(
                f"**Date review:** notebook 05's safer buffers changed the "
                f"minimum observed-sales gap from **{original_gap:,.0f}** to "
                f"**{minimum_gap:,.0f}** units. The largest remaining row would "
                f"have required another **{largest_extra_buffer:,.0f}** buffer "
                "units. Treat this as an event/anomaly override problem rather "
                "than raising every normal-day buffer to that extreme."
            )
        )
    else:
        display(
            Markdown(
                "**Date review:** the recommendation covers all observed sales for "
                "the completed rows. This still does not measure customers lost before a sale was recorded."
            )
        )

    REVIEW_PATH = (
        OUTPUTS_PATH
        / f"{forecast_date:%Y-%m-%d}_inventory_production_review.csv"
    )
    REVIEW_SUMMARY_PATH = (
        OUTPUTS_PATH
        / f"{forecast_date:%Y-%m-%d}_inventory_production_review_summary.csv"
    )
    REVIEW_STORE_PATH = (
        OUTPUTS_PATH
        / f"{forecast_date:%Y-%m-%d}_inventory_production_review_by_store.csv"
    )
    forecast_review.to_csv(REVIEW_PATH, index=False)
    date_summary.to_csv(REVIEW_SUMMARY_PATH, index=False)
    date_by_store.to_csv(REVIEW_STORE_PATH, index=False)
    print("Saved:", REVIEW_PATH)
    print("Saved:", REVIEW_SUMMARY_PATH)
    print("Saved:", REVIEW_STORE_PATH)


## 8. Interpretation limits

- This notebook evaluates production against **observed sales**. It cannot count customers who wanted a sold-out item but left without purchasing.
- The safer buffer is calibrated from out-of-fold FIFO simulations, but the same history is used to choose the buffer. It is a practical service constraint, not a future guarantee.
- `PASTRY_SAFER_SERVICE_TARGET=0.98` is the default compromise. A higher target reduces historical stockouts but increases production, closing stock, and future expiry risk.
- `BaseForecastDemand` is the demand-model prediction. `ForecastDemand` includes the safety buffer and is an inventory target, so it should not be reported as pure model accuracy.
- A product left at closing is not automatically a loss; newly produced stock may be sold the following day.
- `ExpectedFIFOLoss` counts opening old stock that should expire when observed sales are insufficient to clear it.
- Recorded loss can also include transportation damage, handling damage, FIFO mistakes, or counting/entry errors.
- Direct comparison uses only rows with clean inventory movement and complete required actual fields.
- Promotions, anniversaries, half-price days, and other exceptional demand should use an event feature or manual override. A normal-day buffer should not be inflated enough to cover every extreme event.
- A positive estimated margin change does not justify missing already observed sales. Service risk and loss reduction must be reviewed together.
